# Chapter 10: Execution Control: needs, Matrix, Concurrency, Timeouts (notebook edition)

## Learning Objectives

- Predict needs and matrix behavior
- Read fail-fast and concurrency outcomes
- Measure timeout lateness

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

GHA_MODE = os.environ.get("GHA_MODE", "fixture")  # "fixture" | "live"
GHA_REPO = os.environ.get("GHA_REPO", "Friend09/practice_git_intro_to_github_actions")
print("mode:", GHA_MODE, "| repo:", GHA_REPO)

## 1. `needs` propagation

When `a` fails, `b` and `c` are skipped; always/failure jobs still run.

In [ ]:
import json
d = json.loads((ROOT/"fixtures"/"exec_ch10_all.json").read_text())
def needs_results(graph, failed):
    out = {}
    for job in graph:
        out[job] = "failure" if job in failed else ("skipped" if any(out[n] != "success" for n in graph[job]) else "success")
    return out
print(needs_results({"a": [], "b": ["a"], "c": ["b"]}, {"a"}))
real = {j["name"]: j["conclusion"] for j in d["needs"]["jobs"]}
print(real)
assert (real["b"], real["c"], real["report_always"]) == ("skipped", "skipped", "success")

## 2. Matrix expansion

2 x 3 = 6, minus 1 exclude, plus 1 new include gives 6 legs.

In [ ]:
from intro_gha.matrix import expand_matrix
m = d["matrix"]["definition"]
legs = expand_matrix({k: m[k] for k in ("os", "py", "exclude", "include")})
print(len(legs), legs[-1])
assert len(legs) == 6

## 3. fail-fast, before and after

With `true`, two legs never got a runner; with `false`, all six ran. We bill each leg that ran at one minute.

In [ ]:
from datetime import datetime
F = "%Y-%m-%dT%H:%M:%SZ"
ff, nf = d["matrix"]["fail_fast_true"]["jobs"], d["matrix"]["fail_fast_false"]["jobs"]
print(sum(j["ran"] for j in ff), sum(j["ran"] for j in nf))
assert (sum(j["ran"] for j in ff), sum(j["ran"] for j in nf)) == (4, 6)
spans = [(datetime.strptime(j["started_at"], F), datetime.strptime(j["completed_at"], F)) for j in nf]
print("max overlap:", max(sum(s <= t < e for s, e in spans) for t, _ in spans))
assert max(sum(s <= t < e for s, e in spans) for t, _ in spans) == 2

## 4. Concurrency outcomes

The middle run is replaced while pending in both modes.

In [ ]:
x = {r["tag"]: r["run"]["conclusion"] for r in d["concurrency"]["cancel_false"]["runs"]}
y = {r["tag"]: r["run"]["conclusion"] for r in d["concurrency"]["cancel_true"]["runs"]}
print(x, y)
assert x["x2"] == y["y2"] == "cancelled" and x["x3"] == y["y3"] == "success"

## 5. Timeouts fire late

A 60-second job timeout ended the job at 90 seconds, as `cancelled`.

In [ ]:
j = {j["name"]: j for j in d["timeouts"]["jobs"]}["job_timeout"]
secs = int((datetime.strptime(j["completed_at"], F) - datetime.strptime(j["started_at"], F)).total_seconds())
print(j["conclusion"], secs)
assert j["conclusion"] == "cancelled" and secs == 90

## Takeaways & Next Steps

- Skips propagate; `always()` overrides.
- `fail-fast: true` saved two legs; concurrency keeps the newest pending run.
- Timeouts are enforced late.
- Next: Phase 3, Chapter 11 (CI for Python).

## Chapter Link

Read: `learning_modules/chapter_10_execution_control.md`